# 核子-核子散乱問題を解いてみよう
ここではLippmann-Schwinger方程式を用いて核子-核子散乱問題の解法をハンズオンで学びます．
このコース全体では核子ー核子散乱から，原子核の構造計算までを扱う予定です．しかし，最近の現実的な計算では3核子相互作用が不可欠となっています．
3核子相互作用を含む計算は非常に複雑であるため，このコースではあまり触れない予定です．
なので，2核子間相互作用だけでも軽い原子核をそこそこ記述できる${\rm N^{2}LO_{opt}}$ポテンシャルを用います．
まずはpotentialの定義・計算から始めます．
ディレクトリscatteringのimportから始めましょう．

In [ ]:
from pathlib import Path
import os, sys

REPO_NAME = "Lecture_Hokkaido"
GITHUB_REPO = "https://github.com/Takayuki-Miyagi/Lecture_Hokkaido.git"
PACKAGE_DIR = "."

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not Path(REPO_NAME).exists():
        !git clone {GITHUB_REPO}
    %cd {REPO_NAME}
    %pip install -q -e .

def find_repo_root(package_dir=PACKAGE_DIR):
    cwd = Path.cwd().resolve()

    for p in [cwd, *cwd.parents]:
        if (p / "src" / package_dir).is_dir():
            return p

    raise RuntimeError(
        f"Could not find src/{package_dir}. "
        "Please run this notebook from inside the repository."
    )

repo_root = find_repo_root()
src_dir = repo_root / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print("repo root:", repo_root)
print("src added:", src_dir)


import numpy as np
import constants 
from partial_wave_basis import PartialWaveChannels
from nn_pot import NNPotential 
from phaseshift import compute_S_matrix_nn, cross_section_from_S

* まずは下のように``PartialWaveChannels``クラスを用いて、部分波基底を定義します．
ここで，部分波基底は$|k; (LS)JT\rangle$で表されることを思い出してください．
ここで，$k$は相対運動量，$L$は軌道角運動量，$S$はスピン，$J$は全角運動量，$T$はアイソスピンを表します．
ここでは$J_{\rm max}=4$までの部分波を考え、相対運動量のメッシュ点数を$N_{\rm Mesh}=40$、最大相対運動量を$k_{\rm max}=4$ fm$^{-1}$とします．

In [ ]:
pw = PartialWaveChannels(Jmax=4, NMesh=40, kmax=6)

ここで，``pw``は部分波基底を表すクラスのインスタンスで，$J_{\rm max}=4$までの部分波基底のクラス``PartialWaveChannel``をリストで保持しています．
下のコードを実行すると、定義した部分波基底の量子数が表示されます．
また，各チャネルには角運動量$L$，運動量とそのメッシュが保持されています．

In [ ]:
for idx, channel in enumerate(pw.Channels):
    if(idx > 10): continue
    print(f"J={channel.J:2d}, Parity={channel.Parity:2d}, S={channel.S:2d}, Tz={channel.Tz:2d}")
    if(idx==0):
        for i in range(channel.n_dim):
            k, w, L, S, J, Tz = channel.get_quantum_numbers(i)
            print(f"k = {k:7.4f} fm-1, w = {w:8.3e} fm-1, L = {L:3d}, S = {S:3d}, J = {J:3d}, Tz = {Tz:3d}")

さらに，具体的にchiral EFTに基づく核子-核子相互作用のポテンシャルを計算してみましょう．
まずは，``NNPotential``クラスのインスタンスを生成します．
このクラスには``set_potential``メソッドがあり、これを用いてポテンシャルのパラメータを設定・計算します．
``parameters``は辞書型で，計算するポテンシャルのパラメータを指定します．
例えば，A. A. Ekstr&ouml;mらによる${\rm N^{2}LO}$での最適化を行なった${\rm N^{2}LO_{opt}}$ポテンシャル[A. Ekstr&ouml;m et al., Phys. Rev. Lett. 110, 192502 (2013)]を計算する場合は、すでにパラメータセットがLECs.pyの中で定義されているので，``parameters = LECs.N2LO_opt``とします．
計算の詳細に関するパラメータとLow-energy constant(LEC)が定義されています．



In [ ]:
from src import LECs
parameters = LECs.N2LO_opt
for par_name, par_val in parameters.items():
    print(f"{par_name}: {par_val}")

この``parameters``を``NNPotential``クラスの``set_potential``メソッドに渡すと、指定したパラメータに基づいてポテンシャルの計算が行われます．
``python``は激重なので，実行には結構時間がかかります．4, 5分程度は待ちましょう．

また，もう一度同じ計算をしたくないので，計算したポテンシャルはファイルに保存しておきましょう．
ここでは，計算したポテンシャルを``N2LOopt.npz``というファイルに保存します．
このファイルには部分波基底の量子数や相対運動量のメッシュ，計算したポテンシャルの値が保存されます．
このファイルを読み込むことで，次回以降はポテンシャルの計算をスキップして，すぐに散乱問題の解法に進むことができます．

In [ ]:
filename = "N2LOopt.npz"
nnpot = NNPotential(pw)
if(os.path.exists(filename)):
    nnpot.load_npz_into(filename)
else:
    nnpot.set_potential(parameters)
    nnpot.save_npz(filename)    

ここまででポテンシャルの計算と格納が完了しました．まずはポテンシャルが運動量空間でどのような形をしているのかみてみましょう．
ここでは，例としてpnの$^1S_0$チャネルのポテンシャルをプロットしてみます．
このチャネルは軌道角運動量$L=0$(パリティは$(-1)^{L}$なので正)，スピン$S=0$，全角運動量$J=0$，アイソスピンの$z$成分が$Tz=0$のチャネルです．

この図をみると確かに，ポテンシャルは$k=k'=0$のところで最も強い引力的な寄与を持ち，$k$や$k'$が大きくなるにつれて，regulatorによる効果で相互作用が弱くなっていることがわかります．


In [ ]:
import copy
from scipy.interpolate import griddata
from matplotlib import colors
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib import pyplot as plt
J, Parity, S, Tz = 0, 1, 0, 0

fig, ax = plt.subplots(ncols=1, nrows=1)
vmin, vmax = -20, 20 # min and max value for colormap

channel = pw.get_channel(J, Parity, S, Tz)
v = copy.copy(nnpot.vpot[(J, Parity, S, Tz)]) # MeV-2
v *= constants.hc**3 # MeV fm3
NMesh = channel.NMesh
vplot = np.reshape(v[:NMesh, :NMesh], NMesh*NMesh)
Mesh = channel.kmesh[:NMesh]
Meshes = np.reshape(np.tile(Mesh,NMesh), (NMesh,NMesh))
x = Meshes.reshape(NMesh*NMesh)
y = np.transpose(Meshes).reshape(NMesh*NMesh)
X=np.arange(np.min(x), np.max(x), 0.05)
Y=np.arange(np.min(y), np.max(y), 0.05)
Z=griddata((x,y), vplot, (X[None,:], Y[:,None]), method='cubic')
im = ax.imshow(Z, norm=colors.Normalize(vmin=vmin, vmax=vmax), \
        extent=(np.min(x), np.max(x), np.max(y), np.min(y)), cmap="seismic")
divider=make_axes_locatable(ax)
cax = divider.append_axes('right','5%',pad='3%')
cbar = fig.colorbar(im,ax=ax, cax=cax)
cbar.set_label(r"$V(k',k)$ (MeV fm$^{3}$)")
ax.set_xlabel("$k$ (fm$^{-1}$)")
ax.set_ylabel("$k'$ (fm$^{-1}$)")
plt.show()



次に，ポテンシャルを用いてLippmann-Schwinger方程式を解いてみましょう．
ここでは，``compute_S_matrix_nn``関数を用いて$S$行列を計算します．
この$S$行列は
$$
S_{J} = \delta_{LL'} - 2\pi i \mu k T^{SJ}_{LL'}(k, k; E_{k})
$$
で与えられます．ここで，$T^{SL}_{LL'}(k,k; E_{k})$はLippmann-Schwinger方程式の解である$T$行列のオンシェル成分を表します．
Lippmann-Schwinger方程式は$E = \frac{k_{0}^{2}}{2\mu}$として
$$
T^{SJ}_{LL'}(k, k'; E) = V^{SJ}_{LL'}(k, k') + \sum_{L''} \int_0^\infty q^2 dq V^{SJ}_{LL''}(k, q) \frac{2\mu}{k^{2}_{0} - q^2 + i\epsilon} T^{SJ}_{L''L'}(q, k'; E)
$$
で与えられています．
この方程式を数値的に解くためには、上の方程式を
$$
T = V + V G_{0} T
$$
のような行列方程式の形に書き換える方法があります．そうすれば，$T$行列は
$$
T = (1 - V G_{0})^{-1} V
$$
として計算できます．
ここで，$V$はすでに上で確認したように行列っぽく見えています．
$G_{0}$については対角行列として，
$$
G_{0}(i,i) = \begin{cases}
\frac{2 \mu k_{i}^{2} w_{i}}{k_{0} - k_{i}^{2} + i\epsilon} & \text{if } i \neq 0 \\
-i\pi \mu k_{i}  - 2\mu \sum_{j=1}^{N} \frac{k_{i}^{2}w_{j}}{k_{i}^{2} - k_{j}^{2}} & \text{if } i = 0
\end{cases}
$$
となります．ここで，$w_{i}$は相対運動量のメッシュ点に対応する重みです．
このようにして$T$行列を計算したら，あとは上の式を用いて$S$行列を計算するだけです．
具体的な実装に興味のある人は``scattering/phaseshift.py``の``compute_S_matrix_nn``をのぞいてみましょう．

In [ ]:
Smatrices = {}
Elabs = [1, 5, 10] + list(range(20,100,10)) + list(range(100, 350, 50))
mu = [0.5 * constants.m_p, constants.m_p * constants.m_n / (constants.m_p + constants.m_n), 0.5 * constants.m_n]
for channel in pw.Channels:
    J, Prty, S, Tz = channel.J, channel.Parity, channel.S, channel.Tz
    print(f"Doing for J={J:2d}, Parity={Prty:2d}, S={S:2d}, Tz={Tz:2d}")
    for Elab in Elabs:
        if(Tz==-1): q0 = np.sqrt(2.0 * mu[Tz+1]**2 / constants.m_p * Elab) / constants.hc
        if(Tz== 0): q0 = np.sqrt(2.0 * mu[Tz+1]**2 / constants.m_n * Elab) / constants.hc # neutron incident beam
        if(Tz== 1): q0 = np.sqrt(2.0 * mu[Tz+1]**2 / constants.m_n * Elab) / constants.hc
        Smatrices[(J,Prty,S,Tz,Elab)] = compute_S_matrix_nn(nnpot.vpot[(J,Prty,S,Tz)], q0, mu[Tz+1], channel)

# 

これでon-shell $S$行列の計算が完了しました．
これを使ってphase shiftを計算・プロットしてみましょう．
$S$行列からphase shiftを計算することができます．
phase shiftは軌道角運動量の混合がない場合は
$$
\delta_{J} = \frac{1}{2}{\rm arg}(S_{J})
$$
で与えられます．軌道角運動量の混合がある場合は，
$$
S_{J} = \begin{pmatrix}S_{LL} & S_{LL'} \\
S_{L'L} & S_{L'L'} \end{pmatrix} = 
\begin{pmatrix}e^{2i\delta_{L}} \cos 2\epsilon & i e^{i(\delta_{L}+\delta_{L'})} \sin 2\epsilon \\
i e^{i(\delta_{L}+\delta_{L'})} \sin 2\epsilon & e^{2i\delta_{L'}} \cos 2\epsilon \end{pmatrix}
$$
の関係で$\delta_{L}$, $\delta_{L'}$，mixing angle $\epsilon$を計算することができます．


In [ ]:
#phase shifts
phase_shifts = {}
for channel in pw.Channels:
    J, Prty, S, Tz = channel.J, channel.Parity, channel.S, channel.Tz
    for Elab in Elabs:
        Smat = Smatrices[(J,Prty,S,Tz,Elab)]
        if(Smat.shape[0]==1):
            delta = 0.5 * np.angle(Smat[0,0])
            #print(f"J={J:2d}, Prty={Prty:2d}, S={S:2d}, Tz={Tz:2d}, Elab={Elab:6.2f} MeV, delta={delta*180/np.pi:8.4f}")
            phase_shifts[(J,Prty,S,Tz,Elab)] = delta
        else:
            Sigma = 0.5 * np.angle(np.linalg.det(Smat))
            phase = np.exp(-1j * Sigma)
            sin2eps_complex = -1j * Smat[0,1] * phase
            sin2eps = np.real(sin2eps_complex)   
            eps = 0.5 * np.arcsin(sin2eps)
            delta_m = 0.5 * np.angle(Smat[0,0]/np.cos(2*eps))
            delta_p = 0.5 * np.angle(Smat[1,1]/np.cos(2*eps))        
            phase_shifts[(J,Prty,S,Tz,Elab)] = (delta_m, delta_p, eps)  


fig, axs = plt.subplots(ncols=4, figsize=(20,5))
J, Prty, S, Tz = 0, 1, 0, 0
delta = np.array([phase_shifts[(J,Prty,S,Tz,Elab)] for Elab in Elabs])
axs[0].plot(Elabs, delta*180/np.pi, label=f"J={J}, Parity={Prty}, S={S}, Tz={Tz}", ls="-", c='r')

J, Prty, S, Tz = 1, 1, 1, 0
delta_L = np.array([phase_shifts[(J,Prty,S,Tz,Elab)][0] for Elab in Elabs])
delta_Lp = np.array([phase_shifts[(J,Prty,S,Tz,Elab)][1] for Elab in Elabs])
axs[1].plot(Elabs, (np.unwrap(2*delta_L)/2 + np.pi) * 180/np.pi, label=f"J={J}, Parity={Prty}, S={S}, Tz={Tz}, delta_L", ls="-", c='r')
axs[2].plot(Elabs, delta_Lp * 180/np.pi, label=f"J={J}, Parity={Prty}, S={S}, Tz={Tz}, delta_Lp", ls="-", c='r')

J, Prty, S, Tz = 0, -1, 1, 0
delta = np.array([phase_shifts[(J,Prty,S,Tz,Elab)] for Elab in Elabs])
axs[3].plot(Elabs, (np.unwrap(2*delta)/2)*180/np.pi, label=f"J={J}, Parity={Prty}, S={S}, Tz={Tz}", ls="-", c='r')

axs[0].set_ylabel(r"Phase shift (degrees)")
for i in range(len(axs)):
    axs[i].set_xlabel(r"$E_{\rm lab}$ (MeV)")
    axs[i].set_xlim(0,300)
    axs[i].legend()
plt.show()  

弾性散乱の断面積は以下のようになります．
ここでは、pnチャネル(アイソスピンの$z$成分が$Tz=0$のチャネル)の断面積を計算してみましょう．
さらに，弾性散乱の断面積は
$$
\sigma = \frac{\pi}{k^{2}} \sum_{J} \frac{2J+1}{4} {\rm Tr} \left[(S_{J}-1)( S_{J}^{\dagger}-1)\right]
$$
で与えられます．具体的な実装に興味のある人は``scattering/phaseshift.py``の``cross_section_from_S``関数を見てみましょう．


In [ ]:
channel_Tz = 0
sigma = []
for channel in pw.Channels:
    J, Prty, S, Tz = channel.J, channel.Parity, channel.S, channel.Tz
    if(channel_Tz != Tz): continue
    for Elab in Elabs:
        if(Tz==-1): q0 = np.sqrt(2.0 * mu[Tz+1]**2 / constants.m_p * Elab) / constants.hc
        if(Tz== 0): q0 = np.sqrt(2.0 * mu[Tz+1]**2 / constants.m_p * Elab) / constants.hc # proton incident beam
        if(Tz== 1): q0 = np.sqrt(2.0 * mu[Tz+1]**2 / constants.m_n * Elab) / constants.hc
        _sigma = cross_section_from_S(Smatrices[(J,Prty,S,Tz,Elab)], q0, J)
        sigma.append(_sigma)

sigma = np.array(sigma)
sigma = sigma.reshape((int(sigma.size)//len(Elabs), len(Elabs))).transpose()
sigma_tot = np.sum(sigma, axis=1)

fig, ax = plt.subplots()
ax.plot(Elabs, sigma_tot * 1.e3, label=f"Tz={channel_Tz}", ls="-", c='r')
ax.set_xlabel(r"$E_{\rm lab}$ (MeV)")
ax.set_ylabel(r"$\sigma_{\rm tot}$ (mb)")
ax.set_yscale("log")
ax.set_xlim(0,300)
ax.legend()
plt.show()

与えれたポテンシャルを対角化することで、deuteronの束縛状態を確認してみましょう．
束縛状態に対するSchr&ouml;dinger方程式は
$$
(T + V)|\psi\rangle = E |\psi\rangle
$$
と書けます．ここで波動関数は
$$
|\psi\rangle = \sum_{i} c_{i} |\tilde{k}_{i} \rangle
$$
と展開することができます．ここで$|\tilde{k}_{i} \rangle$は，離散化された相対運動の状態で，
$$
\sum_{i} |\tilde{k}_{i} \rangle \langle \tilde{k}_{i}| = 1
$$
となるように，$|\tilde{k}_{i}\rangle = \sqrt{w_{i}} |k_{i}\rangle$と定義されます．
そうすると，もとのSchr&ouml;dinger方程式は
$$
\sum_{j} \left[ \frac{k_{i}^{2}}{2\mu} \delta_{ij} + \tilde{V}_{ij} \right] c_{j} = E c_{i}, \quad \tilde{V}_{ij} = k_{i} \sqrt{w_{i}} V(k_{i}, k_{j}) \sqrt{w_{j}} k_{j}
$$
のような行列方程式の形になります．この行列を対角化することで、deuteronの束縛エネルギーと波動関数を計算することができます．
``NNPotential``クラスの``solve_deuteron``メソッドを呼び出すと、deuteronの束縛エネルギーが計算されます．

In [ ]:
nnpot.solve_deuteron()

# Hands-on questions
* ***$^1S_0$チャネルについてのポテンシャルをプロットしたが、他のチャネルについても同様にプロットしてみよう．軌道角運動量を混ぜるチャネルはどうなっている？***
* ***NN-Online(https://nn-online.org/)のデータベースからpartial wave analysis (PWA)の結果をダウンロードして、計算したphase shift，弾面積と一緒に図示・比較してみよう．***
* ***A. Ekstr&ouml;m et al., Phys. Rev. C 97, 024332 (2018)で与えられている$\Delta$-full EFTパラメータセットを使って展開のオーダーごとで$^{1}S_{0}$チャネルの位相差を計算してみよう．LOからNLOで大きな改善がえられるはずであるが，その理由を考察してみよう．$\Delta$-full EFTの相互作用はparametersのkeyとしてparameters["include_delta"]=Trueと指定することで計算できる．***